# Phase 4: Optimization and the training loop

Phase 3 ended with a single `backward()` call filling gradients for every weight and bias.
This phase turns those gradients into *learning*: an optimizer that nudges each parameter
downhill, and the loop that repeats forward → loss → backward → update until the loss drops.

In [1]:
from babygrad.nn.activations import ReLU, Softmax
from babygrad.nn.losses import CCE
from babygrad.nn.modules import Linear, Sequential
from babygrad.nn.optimizers import SGD
from babygrad.tensor import Tensor

## Collecting the parameters

The optimizer needs to know *which* tensors are learnable. Each layer answers for itself via
`parameters()` — `Linear` returns its `weights` and `bias`, while `ReLU` and `Softmax` have
none and return `[]`. `Sequential.parameters()` walks its layers and flattens those into one list.

In [2]:
model = Sequential(
    [
        Linear(input_size=2, output_size=4),
        ReLU(),
        Linear(input_size=4, output_size=3),
        Softmax(),
    ]
)

params = model.parameters()

# two Linear layers × (weights + bias) = 4 tensors; ReLU/Softmax contribute nothing
[(len(params)), [p.shape for p in params]]

[4, [(1, 4), (2, 4), (1, 3), (4, 3)]]

## The optimizer owns the parameters and the learning rate

`SGD` is constructed with the parameter list and a learning rate. Because `zero_grad` and
`step` operate on the *same* list, they live together on the optimizer — one source of truth.

In [3]:
optimizer = SGD(model.parameters())
optimizer.lr = 0.1

# the optimizer holds references to the very same tensors the layers use,
# so updating them in place actually moves the model's weights
optimizer.parameters[0] is model.layers[0].bias

True

## `zero_grad` — why gradients must be reset each step

Gradients **accumulate**: every `backward()` does `grad += ...`. Without a reset, gradients
from previous steps would pile up and corrupt the current update. Only the *parameters* need
clearing — intermediate tensors are rebuilt fresh every forward pass, so they start at zero anyway.

In [4]:
inputs = Tensor([1, 2, 3, 4], shape=(2, 2))
targets = Tensor([0, 1, 0, 1, 0, 0], shape=(2, 3))

y_pred = model.forward(inputs)
loss = CCE().forward(targets, y_pred)
loss.backward()

before = list(model.layers[0].weights.grad)  # populated by backward
optimizer.zero_grad()
after = list(model.layers[0].weights.grad)   # reset to zeros

{"after_backward": before, "after_zero_grad": after}

{'after_backward': [-0.06180747473857967,
  0.0,
  -0.08004123724641776,
  0.0,
  -0.06214356673331418,
  0.0,
  -0.08000023301496842,
  0.0],
 'after_zero_grad': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}

## `step` — the update rule

The core of SGD: for every parameter, move each value *against* its gradient, scaled by the
learning rate — `data -= lr * grad`. The gradient points uphill (toward higher loss), so the
minus sign steps downhill.

Crucially this mutates `.data` **directly**. It does *not* go through the tensor's `__sub__`,
which would graft the update onto the computational graph and make the next `backward()`
try to backprop through the optimizer itself. The update is bookkeeping, not a differentiable op.

In [5]:
# fresh forward/backward so grads reflect the current weights
optimizer.zero_grad()
y_pred = model.forward(inputs)
loss = CCE().forward(targets, y_pred)
loss.backward()

w = model.layers[0].weights
before = list(w.data)
optimizer.step()
after = list(w.data)

# each weight moved by -lr * grad
changes = [round(a - b, 6) for a, b in zip(after, before)]
{"weights_before": [round(v, 4) for v in before],
 "weights_after": [round(v, 4) for v in after],
 "delta": changes}

{'weights_before': [0.0279,
  -0.095,
  -0.045,
  -0.0554,
  0.0473,
  0.0353,
  0.0784,
  -0.0826],
 'weights_after': [0.0341,
  -0.095,
  -0.037,
  -0.0554,
  0.0535,
  0.0353,
  0.0864,
  -0.0826],
 'delta': [0.006181, 0.0, 0.008004, 0.0, 0.006214, 0.0, 0.008, 0.0]}

## One training step — the five beats

Every step is the same sequence, in this order:

1. `zero_grad()` — clear stale gradients
2. `forward()` — predictions
3. loss — reduce to a single scalar
4. `backward()` — fill every parameter's `.grad`
5. `step()` — nudge every parameter downhill

In [6]:
def train_step(model, optimizer, x, y):
    optimizer.zero_grad()
    y_pred = model.forward(x)
    loss = CCE().forward(y, y_pred)
    loss.backward()
    optimizer.step()
    return loss.data[0]

train_step(model, optimizer, inputs, targets)

1.068967026463343

## The training loop — watch the loss fall

Repeating the step over the same batch should drive the loss down monotonically. A decreasing
loss is the end-to-end proof that parameters, `zero_grad`, autograd, and `step` are all wired
correctly — if any link were broken the curve would be flat, rising, or `NaN`.

In [7]:
model = Sequential(
    [
        Linear(input_size=2, output_size=4),
        ReLU(),
        Linear(input_size=4, output_size=3),
        Softmax(),
    ]
)
optimizer = SGD(model.parameters())
optimizer.lr = 0.1

losses = [train_step(model, optimizer, inputs, targets) for _ in range(50)]
[round(losses[0], 5), round(losses[10], 5), round(losses[-1], 5)]

[1.08855, 0.87585, 0.47251]

## Tracking accuracy alongside loss

Loss is the quantity being optimized, but it is not the quantity you care about — you care about
*correct predictions*. Accuracy measures that directly: the fraction of rows whose predicted class
(the argmax of the softmax row) matches the true class (the argmax of the one-hot target).

It is a read-only metric — never backpropagated through — so it just reads the same `y_pred` the
loss already used. As loss falls, accuracy should climb toward `1.0`.

In [8]:
from babygrad.metrics import accuracy

model = Sequential(
    [
        Linear(input_size=2, output_size=4),
        ReLU(),
        Linear(input_size=4, output_size=3),
        Softmax(),
    ]
)
optimizer = SGD(model.parameters())
optimizer.lr = 0.1

history = []
for _ in range(50):
    optimizer.zero_grad()
    y_pred = model.forward(inputs)
    loss = CCE().forward(targets, y_pred)
    acc = accuracy(targets, y_pred)  # measured on the predictions that produced this loss
    loss.backward()
    optimizer.step()
    history.append((round(loss.data[0], 4), acc))

# (loss, accuracy) at the start, middle, and end of training
{"step_0": history[0], "step_25": history[25], "step_49": history[-1]}

{'step_0': (1.0885, 0.5), 'step_25': (0.6348, 0.5), 'step_49': (0.4725, 1.0)}

## The learning-rate knob

`lr` sets the step size. Too small and the loss crawls; too large and the steps overshoot the
minimum and the loss can bounce or diverge. Re-running the loop at a few rates makes the
trade-off concrete — there is no single right value, it depends on the problem.

In [9]:
def final_loss(lr, steps=50):
    m = Sequential(
        [
            Linear(input_size=2, output_size=4),
            ReLU(),
            Linear(input_size=4, output_size=3),
            Softmax(),
        ]
    )
    opt = SGD(m.parameters())
    opt.lr = lr
    last = None
    for _ in range(steps):
        last = train_step(m, opt, inputs, targets)
    return round(last, 5)

{f"lr={lr}": final_loss(lr) for lr in [0.001, 0.01, 0.1, 1.0, 10.0]}

{'lr=0.001': 1.07894,
 'lr=0.01': 0.99061,
 'lr=0.1': 0.47251,
 'lr=1.0': 0.70789,
 'lr=10.0': 2.47108}